# 00 · Prueba de humo: QLoRA en Colab T4

**TraceGit · ISIA-119 · Grupo 3.** Decide el modelo base antes de minar y entrenar (`PLAN.md` §3).

| Criterio | Aprobado si | Dónde se mide |
|---|---|---|
| (i-a) Carga en 4 bits | El modelo carga en NF4 en la T4 sin OOM | Este notebook |
| (i-b) Entrenamiento estable en fp16 | 30 pasos de QLoRA con **pérdida y norma del gradiente finitas** (sin NaN/inf) | Este notebook |
| (ii) Velocidad en la laptop de la demo | ≥ 3 tokens/s con GGUF q4 | `eval/medir_velocidad_ollama.py` (en la laptop) |

Si Gemma 4 E2B-it falla (i-a) o (i-b), se cambia `MODELO = "qwen"`, se vuelve a correr y se documenta el cambio.

**Cómo ejecutarlo:** VS Code → *Select Kernel* → Colab → servidor con **GPU T4** → Python 3. Ejecutar todo en orden.
Al final, la celda de resultados imprime un JSON: copiarlo a `eval/resultados/` del repositorio.

**Datos de esta prueba:** 128 commits reales anteriores al 30-11-2022 de `psf/requests` (Apache-2.0), solo para
probar la mecánica y medir tiempos. **No forman parte del dataset** del proyecto.

## 1. Configuración

In [1]:
MODELO = "gemma"          # "gemma" (seleccionado) o "qwen" (suplente)
COMPUTE_DTYPE = "fp16"    # "fp16" (plan). Si da NaN, probar "fp32" antes de pasar al suplente y anotarlo.

CANDIDATOS = {
    "gemma": dict(repo="google/gemma-4-E2B-it",
                  revision="4adf15748de9e1a0097b6274cee9c15836c41c37",
                  multimodal=True),
    "qwen":  dict(repo="Qwen/Qwen3-4B",
                  revision="1cfa9a7208912126459214e8b04321603b3df60c",
                  multimodal=False),
}
CFG = CANDIDATOS[MODELO]

# Hiperparámetros de PLAN.md §4.3. Solo cambia la acumulación (4 en vez de 16) para que la prueba dure minutos.
SEED = 42
MAX_PROMPT_TOKENS = 1024
MAX_RESP_CHARS = 512
MAX_LEN = 1536
PASOS = 30
GRAD_ACCUM = 4
N_COMMITS = PASOS * GRAD_ACCUM + 8   # 120 para entrenar + 8 de reserva para generar
LORA_R, LORA_ALPHA, LORA_DROPOUT = 16, 32, 0.05
LR = 2e-4

## 2. Dependencias (versiones fijadas)

`torch` se deja el de Colab y se registra su versión. Gemma 4 pide `transformers` ≥ 5.5 (su `generation_config.json` fue creado con 5.5.0.dev0).

In [2]:
%pip install -q "transformers==5.23.0" "peft==0.21.3" "bitsandbytes==0.50.2" "accelerate==1.16.0"

## 3. Entorno

In [3]:
import json, os, platform, random, subprocess, time, math
import numpy as np
import torch, transformers, peft, bitsandbytes, accelerate

assert torch.cuda.is_available(), "No hay GPU: conectar el kernel a un servidor Colab con T4."
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

ENTORNO = {
    "gpu": torch.cuda.get_device_name(0),
    "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2),
    "capacidad_computo": ".".join(map(str, torch.cuda.get_device_capability(0))),
    "bf16_nativo": torch.cuda.get_device_capability(0) >= (8, 0),   # T4 = 7.5 → False
    "python": platform.python_version(),
    "torch": torch.__version__, "cuda": torch.version.cuda,
    "transformers": transformers.__version__, "peft": peft.__version__,
    "bitsandbytes": bitsandbytes.__version__, "accelerate": accelerate.__version__,
}
print(json.dumps(ENTORNO, indent=2))
if "T4" not in ENTORNO["gpu"]:
    print("AVISO: la GPU no es una T4; los tiempos no representan el plan.")

{
  "gpu": "Tesla T4",
  "vram_total_gb": 14.56,
  "capacidad_computo": "7.5",
  "bf16_nativo": false,
  "python": "3.13.16",
  "torch": "2.14.1+cu130",
  "cuda": "13.0",
  "transformers": "5.23.0",
  "peft": "0.21.3",
  "bitsandbytes": "0.50.2",
  "accelerate": "1.16.0"
}


## 4. Datos de la prueba: commits reales de `psf/requests` (anteriores al 30-11-2022)

Mismo formato que el dataset final (`PLAN.md` §1.2): instrucción = archivos + estadísticas + diff truncado; respuesta = mensaje humano del commit.

In [4]:
REPO_DIR = "/content/requests_humo"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/psf/requests.git", REPO_DIR], check=True)

def git(*args):
    return subprocess.run(["git", "-C", REPO_DIR, *args], capture_output=True, text=True,
                          encoding="utf-8", errors="replace", check=True).stdout

hashes = git("log", "--no-merges", "--before=2022-11-30", "--format=%H", "-n", "600").split()
GENERICOS = {"update", "fix", "wip", "typo", "cleanup", "changes", "minor"}

ejemplos = []
for h in hashes:
    msg = git("log", "-1", "--format=%B", h).strip()
    if len(msg.split()) < 3 or msg.lower().strip(". ") in GENERICOS:
        continue
    stat = git("show", "--format=", "--stat=120", h).strip()
    diff = git("show", "--format=", "--no-color", "-U3", h)
    if not diff.strip():
        continue
    ejemplos.append(dict(sha=h, stat=stat, diff=diff, msg=msg[:MAX_RESP_CHARS]))
    if len(ejemplos) >= N_COMMITS:
        break

print(len(ejemplos), "commits;", "ejemplo:", ejemplos[0]["sha"][:10], "|", ejemplos[0]["msg"][:80])
assert len(ejemplos) >= N_COMMITS

128 commits; ejemplo: 192083576d | Replace git protocol URL in docs with an HTTPS URL


## 5. Carga del modelo en 4 bits (NF4, *double quant*)

Las torres de visión y audio de Gemma 4 **no se cuantizan** ni reciben LoRA: el caso es solo texto.

In [5]:
from transformers import AutoTokenizer, BitsAndBytesConfig

dtype_computo = torch.float16 if COMPUTE_DTYPE == "fp16" else torch.float32
SALTAR = ["vision_tower", "audio_tower", "embed_vision", "embed_audio", "lm_head"]
bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=dtype_computo, llm_int8_skip_modules=SALTAR,
)

tok = AutoTokenizer.from_pretrained(CFG["repo"], revision=CFG["revision"])
if tok.chat_template is None:   # respaldo: la plantilla de Gemma 4 vive en chat_template.jinja
    from transformers import AutoProcessor
    tok.chat_template = AutoProcessor.from_pretrained(CFG["repo"], revision=CFG["revision"]).chat_template
assert tok.chat_template, "El tokenizer no trae plantilla de chat."
if CFG["multimodal"]:
    from transformers import AutoModelForMultimodalLM as Cargador
else:
    from transformers import AutoModelForCausalLM as Cargador

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
model = Cargador.from_pretrained(
    CFG["repo"], revision=CFG["revision"], quantization_config=bnb,
    dtype=torch.float16, device_map={"": 0},
)
CARGA = {
    "clase": type(model).__name__,
    "attn_implementation": getattr(model.config, "_attn_implementation", None),
    "segundos_carga": round(time.time() - t0, 1),
    "vram_tras_carga_gb": round(torch.cuda.memory_allocated() / 2**30, 2),
}
print(CARGA)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

{'clase': 'Gemma4ForConditionalGeneration', 'attn_implementation': 'sdpa', 'segundos_carga': 44.8, 'vram_tras_carga_gb': 6.28}


## 6. Plantilla de chat y enmascarado de la pérdida

La pérdida se calcula **solo sobre la respuesta** (`PLAN.md` §4.3). Se verifica que el prompt sea prefijo exacto de la conversación completa y que la respuesta termine en el token de fin de turno del formato IT (`<turn|>` en Gemma 4, `<|im_end|>` en Qwen3).

In [6]:
SYSTEM = ("Eres un asistente que describe cambios de código. Dado un diff de git, escribe un mensaje de commit "
          "claro y fiel al cambio, sin inventar identificadores.")
EXTRA_PLANTILLA = {"enable_thinking": False}
FIN_TURNO = "<turn|>" if MODELO == "gemma" else "<|im_end|>"
AVISOS = set()

def instruccion(e):
    cuerpo = f"Archivos y estadísticas:\n{e['stat'][:2000]}\n\nDiff:\n"
    base = len(tok(cuerpo)["input_ids"])
    ids_diff = tok(e["diff"], add_special_tokens=False)["input_ids"][: max(0, MAX_PROMPT_TOKENS - base - 80)]
    return cuerpo + tok.decode(ids_diff)

def construir(e):
    msgs = [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": instruccion(e)},
            {"role": "assistant", "content": e["msg"]}]
    prompt = tok.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True, **EXTRA_PLANTILLA)
    completo = tok.apply_chat_template(msgs, tokenize=False, **EXTRA_PLANTILLA)
    if completo.startswith(prompt):
        resp = completo[len(prompt):]
    else:                                    # la plantilla reescribe el historial: se arma a mano
        resp = e["msg"] + FIN_TURNO + "\n"
        AVISOS.add("plantilla_sin_prefijo")
    ids_p = tok(prompt, add_special_tokens=False)["input_ids"]
    ids_r = tok(resp, add_special_tokens=False)["input_ids"]   # por separado: frontera exacta
    ids = (ids_p + ids_r)[:MAX_LEN]
    labels = ([-100] * len(ids_p) + ids_r)[:MAX_LEN]
    return {"input_ids": ids, "labels": labels, "prompt": prompt}

datos = [construir(e) for e in ejemplos]
ej = datos[0]
fin_resp = tok.decode([t for t in ej["labels"] if t != -100])
print("Final de la respuesta enmascarada:", repr(fin_resp[-40:]))
assert FIN_TURNO in fin_resp, f"La respuesta no termina en {FIN_TURNO}: el modelo no aprendería a parar."
print("Avisos:", AVISOS or "ninguno")
longitudes = [len(d["input_ids"]) for d in datos]
print("Tokens por ejemplo: media", int(np.mean(longitudes)), "| p95", int(np.percentile(longitudes, 95)),
      "| máx", max(longitudes))

Final de la respuesta enmascarada: 'ol URL in docs with an HTTPS URL<turn|>\n'
Avisos: ninguno
Tokens por ejemplo: media 551 | p95 1015 | máx 1119


## 7. Línea base: generación antes de entrenar (también prueba la inferencia en fp16)

In [7]:
def generar(d, max_new=96):
    model.eval()
    enc = tok(d["prompt"], return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True)
    return tok.decode(out[0][enc["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

reserva = datos[-8:]
entreno = datos[:-8]
t0 = time.time()
salida_base = generar(reserva[0])
print(f"[{time.time()-t0:.1f}s] BASE:", salida_base[:300])
print("HUMANO:", ejemplos[-8]["msg"][:300])

[6.9s] BASE: **feat: Fix documentation and minor code cleanup**

* Updates `docs/dev/todo.rst` to correct a typo regarding "Google AppEngine".
* Replaces the comment `# AppEngine hackiness.` with `# App Engine hackiness.` in `requests/utils.py`.
HUMANO: Google App Engine (#5475)

* Continuity: Google App Engine


## 8. LoRA solo en las lineales del modelo de lenguaje

En Gemma 4, `vision_tower` y `audio_tower` también tienen capas `q_proj`, `v_proj`, etc. Por eso los módulos objetivo se fijan con una expresión regular sobre `language_model`.

No se usa `prepare_model_for_kbit_training`: pasaría a fp32 todos los parámetros no cuantizados, incluidos los *embeddings* por capa de Gemma 4 (262 144 × 35 × 256 ≈ 2,35 B parámetros), y no cabría en la T4.

In [8]:
from peft import LoraConfig, get_peft_model

if hasattr(model, "peft_config"):
    raise RuntimeError("El modelo ya tiene LoRA aplicado (se re-ejecutó esta celda). "
                       "Reiniciar el kernel y ejecutar todo desde la celda 1.")

if CFG["multimodal"]:
    objetivo = r".*language_model\.layers\.\d+\.(self_attn\.(q|k|v|o)_proj|mlp\.(gate|up|down)_proj)"
else:
    objetivo = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

model.activation_checkpointing_enable(activation_checkpointing_kwargs={"use_reentrant": False})
model.enable_input_require_grads()
model.config.use_cache = False

lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                  target_modules=objetivo, bias="none", task_type="CAUSAL_LM")
model = get_peft_model(model, lora)
for n, p in model.named_parameters():      # adaptadores en fp32 para el GradScaler de fp16
    if p.requires_grad:
        p.data = p.data.float()

mods_lora = sorted({n.split(".lora_")[0] for n, _ in model.named_parameters() if "lora_" in n})
assert mods_lora, "No se aplicó LoRA a ningún módulo."
assert not any(("vision" in m) or ("audio" in m) for m in mods_lora), "LoRA cayó en una torre multimodal."
entrenables = sum(p.numel() for p in model.parameters() if p.requires_grad)
LORA_INFO = {"modulos_lora": len(mods_lora), "parametros_entrenables": entrenables}
print(LORA_INFO, mods_lora[:3])

{'modulos_lora': 205, 'parametros_entrenables': 24158208} ['base_model.model.model.language_model.layers.0.mlp.down_proj', 'base_model.model.model.language_model.layers.0.mlp.gate_proj', 'base_model.model.model.language_model.layers.0.mlp.up_proj']


/usr/local/lib/python3.13/dist-packages/transformers/modeling_utils.py:3303: FutureWarning: `is_gradient_checkpointing` is deprecated and will be removed in v6. Use `is_activation_checkpointing` instead.
  warnings.warn(


## 9. 30 pasos de QLoRA con detección de NaN/inf

In [9]:
import inspect
from transformers import Trainer, TrainingArguments, TrainerCallback

acepta_tti = "token_type_ids" in inspect.signature(model.get_base_model().forward).parameters

def collate(lote):
    largo = max(len(x["input_ids"]) for x in lote)
    pad = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    ids = torch.full((len(lote), largo), pad); lab = torch.full((len(lote), largo), -100)
    att = torch.zeros((len(lote), largo), dtype=torch.long)
    for i, x in enumerate(lote):
        n = len(x["input_ids"])
        ids[i, :n] = torch.tensor(x["input_ids"]); lab[i, :n] = torch.tensor(x["labels"]); att[i, :n] = 1
    out = {"input_ids": ids, "attention_mask": att, "labels": lab}
    if acepta_tti:
        out["token_type_ids"] = torch.zeros_like(ids)
    return out

MAX_INF_SEGUIDOS = 5   # en fp16 el GradScaler salta pasos con grad inf mientras ajusta la escala

class Vigia(TrainerCallback):
    """Detiene solo ante una falla real: pérdida NaN/inf, o gradiente inf durante muchos pasos seguidos."""
    def __init__(self): self.log, self.inf_seguidos = [], 0
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs or "loss" not in logs:
            return
        loss, g = logs["loss"], logs.get("grad_norm")
        self.log.append({"paso": state.global_step, "loss": loss, "grad_norm": g})
        g_ok = g is None or math.isfinite(g)
        self.inf_seguidos = 0 if g_ok else self.inf_seguidos + 1
        print(f"paso {state.global_step:>2} | loss {loss:.4f} | grad_norm {g}" + ("" if g_ok else "  (paso saltado por el GradScaler)"))
        if not math.isfinite(loss):
            print("FALLA: pérdida NaN/inf. Se detiene."); control.should_training_stop = True
        elif self.inf_seguidos >= MAX_INF_SEGUIDOS:
            print(f"FALLA: gradiente inf {MAX_INF_SEGUIDOS} pasos seguidos. Se detiene."); control.should_training_stop = True

vigia = Vigia()
args = TrainingArguments(
    output_dir="/content/humo_out", max_steps=PASOS, per_device_train_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="cosine",
    warmup_steps=0.03, optim="paged_adamw_8bit", fp16=(COMPUTE_DTYPE == "fp16"), bf16=False,
    logging_steps=1, save_strategy="no", report_to="none", seed=SEED,
    remove_unused_columns=False, activation_checkpointing=False,  # ya activado arriba en el modelo
)
trainer = Trainer(model=model, args=args, train_dataset=entreno, data_collator=collate, callbacks=[vigia])

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
trainer.train()
seg = time.time() - t0
print(f"{seg/60:.1f} min · pico VRAM {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

Step,Training Loss
1,6.489629
2,6.489692
3,5.699188
4,6.214913
5,6.433572
6,6.814238
7,5.157884
8,5.793419
9,4.699552
10,3.784006


paso  1 | loss 6.4896 | grad_norm nan  (paso saltado por el GradScaler)
paso  2 | loss 6.4897 | grad_norm nan  (paso saltado por el GradScaler)
paso  3 | loss 5.6992 | grad_norm nan  (paso saltado por el GradScaler)
paso  4 | loss 6.2149 | grad_norm 12.884649276733398
paso  5 | loss 6.4336 | grad_norm nan  (paso saltado por el GradScaler)
paso  6 | loss 6.8142 | grad_norm 14.305809020996094
paso  7 | loss 5.1579 | grad_norm 8.659245491027832
paso  8 | loss 5.7934 | grad_norm 10.503338813781738
paso  9 | loss 4.6996 | grad_norm 12.71026611328125
paso 10 | loss 3.7840 | grad_norm 13.717066764831543
paso 11 | loss 4.0628 | grad_norm 9.61496639251709
paso 12 | loss 3.4336 | grad_norm 12.166783332824707
paso 13 | loss 2.3160 | grad_norm 6.710373401641846
paso 14 | loss 3.6948 | grad_norm 7.337367534637451
paso 15 | loss 3.1854 | grad_norm 7.4395647048950195
paso 16 | loss 2.6188 | grad_norm 5.675785541534424
paso 17 | loss 2.2626 | grad_norm 3.8079535961151123
paso 18 | loss 1.9764 | grad_n

## 10. Resultado y decisión

In [10]:
losses = [x["loss"] for x in vigia.log]
grads = [x["grad_norm"] for x in vigia.log if x["grad_norm"] is not None]
grads_finitos = [g for g in grads if math.isfinite(g)]
pasos_saltados = len(grads) - len(grads_finitos)
finitos = (len(losses) == PASOS and all(math.isfinite(v) for v in losses)
           and pasos_saltados <= 0.2 * PASOS)
seg_por_ejemplo = seg / (PASOS * GRAD_ACCUM)
salida_lora = generar(reserva[0])

RESULTADO = {
    "fecha": time.strftime("%Y-%m-%d %H:%M"), "modelo": f"{CFG['repo']}@{CFG['revision']}",
    "compute_dtype": COMPUTE_DTYPE, "entorno": ENTORNO, "carga": CARGA, "lora": LORA_INFO,
    "tokens_por_ejemplo": {"media": int(np.mean(longitudes)), "p95": int(np.percentile(longitudes, 95))},
    "pasos_completados": len(losses), "loss_primeros5": round(float(np.mean(losses[:5])), 4),
    "loss_ultimos5": round(float(np.mean(losses[-5:])), 4), "grad_norm_max_finito": round(max(grads_finitos), 4) if grads_finitos else None,
    "pasos_saltados_gradscaler": pasos_saltados,
    "pico_vram_gb": round(torch.cuda.max_memory_allocated() / 2**30, 2),
    "seg_por_ejemplo": round(seg_por_ejemplo, 2),
    "estimado_horas_3000x2_epocas": round(seg_por_ejemplo * 6000 / 3600, 2),
    "aprobado_i": finitos, "avisos": sorted(AVISOS),
    "salida_base": salida_base[:300], "salida_tras_30_pasos": salida_lora[:300],
    "log": vigia.log,
}
print(json.dumps(RESULTADO, indent=2, ensure_ascii=False))
print("\nDECISIÓN (i):", "APROBADO" if finitos else "FALLA → probar COMPUTE_DTYPE='fp32' o MODELO='qwen'")
if RESULTADO["estimado_horas_3000x2_epocas"] > 11:
    print("AVISO: el entrenamiento completo no cabe en una sesión gratuita (≤12 h): reducir datos/secuencia o usar checkpoints.")

{
  "fecha": "2026-11-18 15:28",
  "modelo": "google/gemma-4-E2B-it@4adf15748de9e1a0097b6274cee9c15836c41c37",
  "compute_dtype": "fp16",
  "entorno": {
    "gpu": "Tesla T4",
    "vram_total_gb": 14.56,
    "capacidad_computo": "7.5",
    "bf16_nativo": false,
    "python": "3.13.16",
    "torch": "2.14.1+cu130",
    "cuda": "13.0",
    "transformers": "5.23.0",
    "peft": "0.21.3",
    "bitsandbytes": "0.50.2",
    "accelerate": "1.16.0"
  },
  "carga": {
    "clase": "Gemma4ForConditionalGeneration",
    "attn_implementation": "sdpa",
    "segundos_carga": 44.8,
    "vram_tras_carga_gb": 6.28
  },
  "lora": {
    "modulos_lora": 205,
    "parametros_entrenables": 24158208
  },
  "tokens_por_ejemplo": {
    "media": 551,
    "p95": 1015
  },
  "pasos_completados": 30,
  "loss_primeros5": 6.2654,
  "loss_ultimos5": 2.5158,
  "grad_norm_max_finito": 14.3058,
  "pasos_saltados_gradscaler": 4,
  "pico_vram_gb": 10.51,
  "seg_por_ejemplo": 1.2,
  "estimado_horas_3000x2_epocas": 2.0,
  "a